In [1]:
import ast
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import Word2Vec
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
W2V_DIM       = 300       # Word2Vec embedding dimension
WORD_HIDDEN   = 64        # Bi-GRU hidden size per direction → 128 total
SENT_HIDDEN   = 64        # Bi-GRU hidden size per direction → 128 total
MAX_WORD_TOK  = 50        # Max tokens per sentence
MAX_SENTENCES = 20        # Max sentences per document
TRIBUNAL_DIM  = 16
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD & CLEAN DATA
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv('../court_cases.csv', parse_dates=['Date'])
df = df.drop(columns=['Coram', 'Judge', 'Date', 'Plaintiff_Name', 'Defendant_Name',
                       'Combined_Rule', 'Combined_Application', 'plaintiff_label'])

In [4]:
df.head()

,Case_Number,Tribunal_Court,Combined_Facts,Combined_Issue,defendant_label
0,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, based on the pre-2003 employment co...",Liable
1,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...",['Whether the pre-2003 conduct of Sher Hock Gu...,Liable
2,Suit 798/2007,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '19...","['Whether, before and up to February 2003, the...",Liable
3,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable
4,Suit No 477 of 2012; Suit No 1015 of 2012; Reg...,High Court,"[[{'Fact_Type': 'PARTY_INFO', 'Fact_Date': '20...","['Whether, in Suit No 477 of 2012, losses to A...",Liable


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. TEXT EXTRACTION HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def parse_if_string(obj):
    if isinstance(obj, str):
        try:
            return ast.literal_eval(obj)
        except Exception:
            return obj
    return obj

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def extract_facts_sentences(raw):
    """
    Returns list of (sentence_str, fact_type_str, fact_date_str) tuples.
    Fact_Type and Fact_Date are prepended to the sentence text so the
    Word2Vec corpus and HAN both see them inline.
    """
    raw = parse_if_string(raw)
    if isinstance(raw, list) and raw and isinstance(raw[0], list):
        flat = [item for sublist in raw for item in sublist]
    else:
        flat = raw if isinstance(raw, list) else [raw]

    sentences = []
    for item in flat:
        item = parse_if_string(item)
        if not isinstance(item, dict):
            continue
        fact_type = clean(item.get("Fact_Type", ""))
        fact_date = clean(item.get("Fact_Date", ""))
        text_parts = [
            clean(v) for k, v in item.items()
            if k not in ("Fact_Type", "Fact_Date") and isinstance(v, str)
        ]
        sentence = " ".join([fact_type, fact_date] + text_parts).strip()
        if sentence:
            sentences.append(sentence)
    return sentences   # list of strings

def extract_issue_sentences(raw):
    """Returns list of sentence strings from Combined_Issue (LoS)."""
    raw = parse_if_string(raw)
    if isinstance(raw, list):
        return [clean(s) for s in raw if str(s).strip()]
    return [clean(str(raw))]

def tokenize(sentence):
    """Splits a cleaned sentence into a list of word tokens."""
    return sentence.split()

# Apply extraction
df["facts_sentences"] = df["Combined_Facts"].apply(extract_facts_sentences)
df["issue_sentences"] = df["Combined_Issue"].apply(extract_issue_sentences)

print(f"Sample facts sentences: {df['facts_sentences'].iloc[0][:2]}")
print(f"Sample issue sentences: {df['issue_sentences'].iloc[0][:2]}")

Sample facts sentences: ['party info 1990 01 01 abb holdings pte ltd is part of the abb group a worldwide group of companies that manufacture market and sell several products for industrial and commercial uses including low medium and high voltage circuit breakers', 'party info 1990 01 01 abb holdings pte ltd is part of the singapore component of the abb group collectively referred to with other singapore entities as the abb singapore group']
Sample issue sentences: ['whether based on the pre 2003 employment contracts and employee handbook the scope of contractual and equitable obligations of senior management personnel extended to restricting their participation in external business ventures and competitive activities involving medium voltage switchgear products', 'whether prior to the commencement of proceedings in december 2007 the sequence of roles and actions undertaken by sher hock guan charles in relation to abb entities xian high voltage apparatus research institute xiamen huad

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. TRAIN WORD2VEC ON FULL CORPUS
#    Corpus = all tokenized sentences from both facts and issue fields
# ─────────────────────────────────────────────────────────────────────────────

all_tokenized_sentences = []
for sents in df["facts_sentences"]:
    for s in sents:
        all_tokenized_sentences.append(tokenize(s))
for sents in df["issue_sentences"]:
    for s in sents:
        all_tokenized_sentences.append(tokenize(s))

print(f"Total sentences for Word2Vec training: {len(all_tokenized_sentences)}")

w2v_model = Word2Vec(
    sentences  = all_tokenized_sentences,
    vector_size= W2V_DIM,
    window     = 5,
    min_count  = 1,
    workers    = 4,
    epochs     = 10,
    sg         = 1        # Skip-gram (better for rare legal terms)
)

vocab      = w2v_model.wv
VOCAB_SIZE = len(vocab)
print(f"Word2Vec vocabulary size: {VOCAB_SIZE}")

# Build embedding matrix: row i = embedding for word index i
# Reserve index 0 for padding
word2idx = {word: idx + 1 for idx, word in enumerate(vocab.index_to_key)}
PAD_IDX  = 0

embedding_matrix = np.zeros((VOCAB_SIZE + 1, W2V_DIM), dtype=np.float32)
for word, idx in word2idx.items():
    embedding_matrix[idx] = vocab[word]

print(f"Embedding matrix shape: {embedding_matrix.shape}")

Total sentences for Word2Vec training: 15866
Word2Vec vocabulary size: 9067
Embedding matrix shape: (9068, 300)


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. TRIBUNAL_COURT  →  16-dim one-hot vector
# ─────────────────────────────────────────────────────────────────────────────

tribunal_enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
tribunal_enc.fit(df[["Tribunal_Court"]])

def get_tribunal_vec(court_val):
    vec = tribunal_enc.transform(
        pd.DataFrame([[court_val]], columns=["Tribunal_Court"])
    )[0]
    if len(vec) < TRIBUNAL_DIM:
        vec = np.pad(vec, (0, TRIBUNAL_DIM - len(vec)))
    else:
        vec = vec[:TRIBUNAL_DIM]
    return vec   # (16,)

tribunal_vecs = np.vstack([get_tribunal_vec(v) for v in df["Tribunal_Court"]])
print(f"Tribunal matrix: {tribunal_vecs.shape}")   # (n_cases, 16)

Tribunal matrix: (424, 16)


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. HAN COMPONENTS  (no classes — functional definitions)
# ─────────────────────────────────────────────────────────────────────────────

# ── Shared embedding layer ───────────────────────────────────────────────────
embedding_layer = nn.Embedding(
    num_embeddings = VOCAB_SIZE + 1,
    embedding_dim  = W2V_DIM,
    padding_idx    = PAD_IDX
)
embedding_layer.weight = nn.Parameter(
    torch.tensor(embedding_matrix), requires_grad=True
)
embedding_layer = embedding_layer.to(DEVICE)

# ── Word Encoder  (2x Bi-GRU, 64 hidden → 128 per token) ────────────────────
word_bigru = nn.GRU(
    input_size    = W2V_DIM,
    hidden_size   = WORD_HIDDEN,
    num_layers    = 2,
    batch_first   = True,
    bidirectional = True,
    dropout       = 0.3
).to(DEVICE)

# ── Word Attention  (MLP + residual + LayerNorm + context vector u_w) ────────
word_attn_mlp  = nn.Linear(WORD_HIDDEN * 2, WORD_HIDDEN * 2).to(DEVICE)
word_attn_norm = nn.LayerNorm(WORD_HIDDEN * 2).to(DEVICE)
word_context   = nn.Parameter(torch.randn(WORD_HIDDEN * 2).to(DEVICE))

# ── Sentence Encoder  (2x Bi-GRU, 64 hidden → 128 per sentence) ─────────────
sent_bigru = nn.GRU(
    input_size    = WORD_HIDDEN * 2,
    hidden_size   = SENT_HIDDEN,
    num_layers    = 2,
    batch_first   = True,
    bidirectional = True,
    dropout       = 0.3
).to(DEVICE)

# ── Sentence Attention  (MLP + residual + LayerNorm + context vector u_s) ────
sent_attn_mlp  = nn.Linear(SENT_HIDDEN * 2, SENT_HIDDEN * 2).to(DEVICE)
sent_attn_norm = nn.LayerNorm(SENT_HIDDEN * 2).to(DEVICE)
sent_context   = nn.Parameter(torch.randn(SENT_HIDDEN * 2).to(DEVICE))

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. HAN FORWARD FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def sentence_to_ids(sentence, max_len=MAX_WORD_TOK):
    """Converts a sentence string to a padded list of word indices."""
    tokens = tokenize(sentence)[:max_len]
    ids    = [word2idx.get(t, PAD_IDX) for t in tokens]
    ids   += [PAD_IDX] * (max_len - len(ids))   # pad
    return ids   # list of ints, length = max_len

def word_attention(h):
    """
    h      : (seq_len, 128)
    returns: (128,)  weighted sentence vector
    """
    u     = word_attn_norm(h + torch.tanh(word_attn_mlp(h)))
    score = torch.matmul(u, word_context)
    alpha = torch.softmax(score, dim=0)
    return torch.matmul(alpha.unsqueeze(0), h).squeeze(0)   # (128,)

def sentence_attention(h):
    """
    h      : (num_sentences, 128)
    returns: (128,)  document vector
    """
    u     = sent_attn_norm(h + torch.tanh(sent_attn_mlp(h)))
    score = torch.matmul(u, sent_context)
    alpha = torch.softmax(score, dim=0)
    return torch.matmul(alpha.unsqueeze(0), h).squeeze(0)   # (128,)

def encode_document(sentences, max_sents=MAX_SENTENCES):
    """
    sentences : list of sentence strings
    Returns   : (128,) document vector via Word Encoder → Word Attn
                → Sentence Encoder → Sentence Attn
    """
    if len(sentences) == 0:
        return torch.zeros(SENT_HIDDEN * 2).to(DEVICE)

    sentences = sentences[:max_sents]

    # ── Word Encoder: encode each sentence → (128,) ──────────────────────────
    sent_vecs = []
    for sent in sentences:
        ids    = sentence_to_ids(sent)                              # (max_word_tok,)
        id_t   = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
        emb    = embedding_layer(id_t)                             # (1, max_word_tok, 300)
        h, _   = word_bigru(emb)                                   # (1, max_word_tok, 128)
        h      = h.squeeze(0)                                      # (max_word_tok, 128)
        sv     = word_attention(h)                                  # (128,)
        sent_vecs.append(sv)

    # Pad to max_sents
    while len(sent_vecs) < max_sents:
        sent_vecs.append(torch.zeros(WORD_HIDDEN * 2).to(DEVICE))

    # ── Sentence Encoder: encode all sentence vectors → (128,) ───────────────
    mat    = torch.stack(sent_vecs, dim=0).unsqueeze(0)            # (1, max_sents, 128)
    h, _   = sent_bigru(mat)                                       # (1, max_sents, 128)
    h      = h.squeeze(0)                                          # (max_sents, 128)
    return sentence_attention(h)                                   # (128,)

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. BUILD FEATURE MATRIX
#    facts_doc(128) | issue_doc(128) | tribunal(16)  →  272-dim per case
# ─────────────────────────────────────────────────────────────────────────────

def build_feature_vector(row, tribunal_vec):
    with torch.no_grad():
        facts_vec    = encode_document(row["facts_sentences"])     # (128,)
        issue_vec    = encode_document(row["issue_sentences"])     # (128,)
    tribunal_t   = torch.tensor(tribunal_vec, dtype=torch.float32).to(DEVICE)
    fused        = torch.cat([facts_vec, issue_vec, tribunal_t], dim=0)  # (272,)
    return fused.cpu().numpy()

print("Building feature matrix...")
X_all = np.vstack([
    build_feature_vector(row, tribunal_vecs[i])
    for i, (_, row) in enumerate(df.iterrows())
])
print(f"Feature matrix shape: {X_all.shape}")   # (n_cases, 272)

Building feature matrix...
Feature matrix shape: (424, 272)


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. LABELS
# ─────────────────────────────────────────────────────────────────────────────

label_map = {'Not Liable': 0, 'Liable': 1}
y_all     = df["defendant_label"].map(label_map).values

print("Label distribution:")
unique, counts = np.unique(y_all, return_counts=True)
for val, count in zip(unique, counts):
    print(f"  {val} ({'Not Liable' if val == 0 else 'Liable'}): {count}")

Label distribution:
  0 (Not Liable): 213
  1 (Liable): 211


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. TRAIN / TEST SPLIT  (group-based to prevent data leakage)
# ─────────────────────────────────────────────────────────────────────────────

case_ids  = df["Case_Number"].values
splitter  = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=case_ids))

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Unique train cases: {len(set(case_ids[train_idx]))}  |  "
      f"Unique test cases:  {len(set(case_ids[test_idx]))}")

Train: (347, 272)  |  Test: (77, 272)
Unique train cases: 109  |  Unique test cases:  28


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 11. GRID SEARCH  (raw LinearSVC — calibration applied after)
# ─────────────────────────────────────────────────────────────────────────────

param_grid = {
    'C':        [0.01, 0.1, 1.0, 10.0],
    'max_iter': [20000, 30000],
    'loss':     ['squared_hinge', 'hinge']
}

grid_search = GridSearchCV(
    LinearSVC(class_weight="balanced", random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

print("Starting GridSearchCV...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters:       {grid_search.best_params_}")
print(f"Best CV F1 (weighted): {grid_search.best_score_:.4f}")

results_df = pd.DataFrame(grid_search.cv_results_)
print("\nTop 5 parameter combinations:")
print(results_df[['param_C', 'param_max_iter', 'param_loss', 'mean_test_score']]
      .sort_values('mean_test_score', ascending=False)
      .head())

Starting GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best parameters:       {'C': 0.1, 'loss': 'hinge', 'max_iter': 20000}
Best CV F1 (weighted): 0.6030

Top 5 parameter combinations:
   param_C  param_max_iter     param_loss  mean_test_score
6     0.10           20000          hinge         0.602998
7     0.10           30000          hinge         0.602998
4     0.10           20000  squared_hinge         0.591812
5     0.10           30000  squared_hinge         0.591812
1     0.01           30000  squared_hinge         0.576430


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 12. CALIBRATE BEST MODEL
# ─────────────────────────────────────────────────────────────────────────────

best_svm = LinearSVC(
    **grid_search.best_params_,
    class_weight="balanced",
    random_state=42
)

best_model = CalibratedClassifierCV(best_svm, cv=5)
best_model.fit(X_train, y_train)
print("Calibrated best model fitted.")


Calibrated best model fitted.


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 13. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

label_names  = {0: 'Not Liable', 1: 'Liable'}
target_names = [label_names[0], label_names[1]]

print(classification_report(y_test, y_pred, target_names=target_names))

proba_df = pd.DataFrame(y_proba, columns=[f"P({n})" for n in target_names])
proba_df["Predicted"]  = [label_names[p] for p in y_pred]
proba_df["True_Label"] = [label_names[t] for t in y_test]
print("\nPrediction probabilities:")
print(proba_df.head(10))

              precision    recall  f1-score   support

  Not Liable       0.61      0.51      0.56        43
      Liable       0.49      0.59      0.53        34

    accuracy                           0.55        77
   macro avg       0.55      0.55      0.55        77
weighted avg       0.56      0.55      0.55        77


Prediction probabilities:
   P(Not Liable)  P(Liable)   Predicted  True_Label
0       0.475749   0.524251      Liable      Liable
1       0.453610   0.546390      Liable      Liable
2       0.592418   0.407582  Not Liable      Liable
3       0.604400   0.395600  Not Liable      Liable
4       0.399124   0.600876      Liable  Not Liable
5       0.401859   0.598141      Liable  Not Liable
6       0.605052   0.394948  Not Liable      Liable
7       0.590201   0.409799  Not Liable      Liable
8       0.592064   0.407936  Not Liable  Not Liable
9       0.610289   0.389711  Not Liable  Not Liable
